# Stage LR — Generation Backbone Comparison

**Exploratory Stage LR research only — does not touch `main`, per the architecture freeze (`VALIDATION.md` §56).** This notebook does not train anything. It reuses the project's own **already-validated** phoneme-constrained decoding mechanism (`rephrase.py::PhonemeConstraintLogitsProcessor` — measured to take leak-free generation from 4%→100% and usable-candidate rate from ~9-13%→52%, `VALIDATION.md` §36.2/§52) and asks a narrower question: **does swapping in a different/larger base generation model, now that GPU compute is available, produce better candidates than the currently-deployed one** — without touching the decoding-time constraint logic itself, which already works.

**What's already known, so this doesn't repeat a settled test:**

| Prior test | Result | Why this notebook is still worth running |
|---|---|---|
| R21: `flan-t5-base` as an escalation-tier rewriter | Best similarity score seen, 0.950 | Never tested combined with the phoneme constraint specifically — worth checking directly, not assumed to transfer |
| R23: Qwen2.5-0.5B/1.5B-Instruct as rewriters, on CPU | 0.5B: 0/8 passed, worse than baseline. 1.5B: better task-following but stilted, one factual error, both 10-40x slower than T5 on CPU | The speed penalty (the practical reason small decoder-only models looked bad) is a **CPU-only** artifact — this notebook removes it via GPU, and tests **larger** models than were previously practical to even try, not a re-run of the same negative result |

**Honest disclosure:** written and reviewed carefully, not executed — no GPU/Colab access in the environment that wrote it. Expect first-run debugging.

**Quality judging happens back in the main session, not here.** This notebook produces raw candidate outputs and hard mechanical checks (did a blocked sound actually leak through — yes/no, checkable by code, no judgment call). It deliberately does **not** attempt to score meaning-preservation or naturalness itself; download the results JSON at the end and it gets blind-judged the same way every other batch in this project has been, via a fresh Claude call with no metadata.

## 0. Setup — GPU runtime required

**Runtime > Change runtime type > T4 GPU (or better)** before running, or the larger model in section 3 will not fit / will be impractically slow.

In [ ]:
# IMPORTANT: torch/torchvision are deliberately NOT upgraded here. Colab
# ships a matched, working torch+torchvision pair; upgrading torch alone
# (a mistake in an earlier version of this notebook) desyncs torchvision's
# compiled ops from it, and transformers' auto-loading machinery then fails
# with a confusing "ModuleNotFoundError: Could not import module
# 'Qwen2ForCausalLM'" that actually masks a "torchvision::nms does not
# exist" RuntimeError underneath. Only pure-Python libraries are touched.
#
# wordfreq is required transitively: rephrase.py -> semantic.py -> freq.py
# -> wordfreq.zipf_frequency (project's own memory-safe wrapper). Missing
# it breaks the `import rephrase` cell below with an unrelated-looking
# ModuleNotFoundError.
!pip install -q -U transformers accelerate sentencepiece nltk wordfreq

In [ ]:
# Sanity check torch/torchvision are actually paired correctly BEFORE
# anything else runs, so a mismatch surfaces here clearly instead of deep
# inside a Qwen2ForCausalLM import failure later.
import torch
print("torch:", torch.__version__)
try:
    import torchvision
    print("torchvision:", torchvision.__version__)
    _ = torch.ops.torchvision.nms  # the exact op that broke before -- probe it directly
    print("torch/torchvision compiled ops OK.")
except Exception as e:
    raise RuntimeError(
        "torch/torchvision mismatch detected (this is the failure mode that broke "
        "an earlier run of this notebook). Do NOT try to fix by upgrading torch "
        "alone. Instead: Runtime > Disconnect and delete runtime, then Runtime > "
        "Run all on a fresh instance -- Colab's preinstalled pair should be "
        "consistent on a clean VM."
    ) from e

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU detected -- set Runtime > Change runtime type > GPU, then re-run."
print("GPU:", torch.cuda.get_device_name(0))

## 1. Pull the already-validated pieces directly from the repo

No participant data involved here — `rephrase.py`, `phonetic.py`, and `eval/step3_gencheck_corpus.py` (the fresh generalization-check corpus, `VALIDATION.md` §54) are all tracked, public, non-sensitive files.

In [ ]:
!rm -rf speech-ai
!git clone --branch stage-lr --depth 1 https://github.com/haqiqak/speech-ai.git
import sys
sys.path.insert(0, "speech-ai")

In [ ]:
import nltk
# wordnet/omw-1.4 added defensively: semantic.py (imported transitively by
# rephrase.py) does `from nltk.corpus import wordnet as wn` at module level
# and instantiates a WordNetLemmatizer -- cheap to pre-download rather than
# risk a lazy-load failure mid-run.
for pkg in ["cmudict", "wordnet", "omw-1.4", "punkt", "punkt_tab",
            "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
    try:
        nltk.download(pkg, quiet=True)
    except Exception as e:
        print(f"skip {pkg}: {e}")

In [ ]:
import os
os.environ["REPHRASE_DEVICE"] = "cuda"

import importlib
import phonetic
import rephrase
from rephrase import PhonemeConstraintLogitsProcessor

print("imported rephrase.py / phonetic.py from the tracked repo directly -- not reimplemented.")

## 2. Eval material — the tracked fresh-corpus check, a small subset for speed

Reusing `eval/step3_gencheck_corpus.py`'s own (sentence, profile) pairs — the same material `VALIDATION.md` §54's 21.4% fresh-corpus figure came from — so any difference found here is directly comparable to an existing number, not a new, incomparable sample.

In [ ]:
sys.path.insert(0, "speech-ai/eval")
from step3_gencheck_corpus import CORPUS, RUN_PLAN

by_id = {c["id"]: c for c in CORPUS}

# Subset for speed -- running 3 backbone models across all 36 would be slow.
# Take every 3rd dense_mixed run plus every 3rd core_word run (dense_mixed is
# where the frozen system's own numbers show 0% CLEAN, VALIDATION.md SS54 --
# the most informative slice to re-check with a different generator).
dense = [r for r in RUN_PLAN if r["profile_type"] == "dense_mixed"][::3]
core = [r for r in RUN_PLAN if r["profile_type"] == "core_word"][::3]
SUBSET = dense + core
print(f"using {len(SUBSET)} of {len(RUN_PLAN)} runs ({len(dense)} dense_mixed, {len(core)} core_word)")

def sentence_and_blocked(run):
    sent = by_id[run["id"]]["text"]
    blocked_words = run.get("words", [])
    blocked_patterns = run.get("sounds", [])
    return sent, blocked_words, blocked_patterns

## 3. Backbones to compare

1. **Current production default** — `Vamsi/T5_Paraphrase_Paws` (control; whatever this notebook finds should be measured against this, not in isolation).
2. **`flan-t5-base`** — scored best on generic rewrite quality in R21, never tested with the phoneme constraint specifically.
3. **A GPU-scale instruct model** — `Qwen/Qwen2.5-3B-Instruct`. R23 already found the 0.5B size performs badly and the 1.5B size is slow-but-more-capable on CPU; this asks the actual open question (does scale help) rather than re-confirming the small-size result.

In [ ]:
SEQ2SEQ_BACKBONES = ["Vamsi/T5_Paraphrase_Paws", "google/flan-t5-base"]  # bare "flan-t5-base" 404s -- needs the org prefix
CAUSAL_BACKBONE = "Qwen/Qwen2.5-3B-Instruct"

In [ ]:
def run_seq2seq_backbone(model_name, runs, k=5):
    """Reuses generate_candidates_phoneme_constrained() UNMODIFIED --
    only the module-level model pointer is swapped between calls."""
    os.environ["REPHRASE_MODEL"] = model_name
    # Reset rephrase.py's lazy-load state so it picks up the new model name
    # instead of reusing whatever loaded first.
    rephrase._model = None
    rephrase._tokenizer = None
    rephrase._load_tried = False
    rephrase._rephrase_ok = False
    rephrase.REPHRASE_MODEL = model_name

    results = []
    for run in runs:
        sent, blocked_words, blocked_patterns = sentence_and_blocked(run)
        import time
        t0 = time.perf_counter()
        candidates, stats = rephrase.generate_candidates_phoneme_constrained(
            sent, k=k, blocked_words=blocked_words, blocked_patterns=blocked_patterns,
        )
        latency = time.perf_counter() - t0
        results.append({
            "run_id": run["id"], "original": sent,
            "blocked_words": blocked_words, "blocked_patterns": blocked_patterns,
            "candidates": candidates, "stats": stats, "latency_s": round(latency, 3),
        })
    return results


def _load_causal_model(model_name):
    """dtype= replaced torch_dtype= in newer transformers releases; try the
    current name first and fall back for older pinned versions, rather than
    hardcoding one and breaking on whichever transformers version Colab
    actually resolves."""
    from transformers import AutoModelForCausalLM
    try:
        return AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.float16, device_map="cuda")
    except TypeError:
        return AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map="cuda")


def run_causal_backbone(model_name, runs, k=5):
    """Adapts the SAME PhonemeConstraintLogitsProcessor to a causal LM's
    generate() call. Key difference from the seq2seq path: input_ids during
    generation includes the full prompt, not just the decoder's own output,
    so decoder_start_len must be the prompt's token length -- otherwise the
    processor could flag a blocked word that legitimately appears in the
    INSTRUCTION/PROMPT text itself (e.g. naming the word to avoid), not in
    the model's actual output."""
    from transformers import AutoTokenizer, LogitsProcessorList
    tok = AutoTokenizer.from_pretrained(model_name)
    model = _load_causal_model(model_name)

    results = []
    for run in runs:
        sent, blocked_words, blocked_patterns = sentence_and_blocked(run)
        avoid_note = ""
        if blocked_words:
            avoid_note += f" Do not use these words: {', '.join(blocked_words)}."
        messages = [{
            "role": "user",
            "content": f"Reword this sentence so it means exactly the same thing, in natural English. "
                       f"Output only the reworded sentence, nothing else.{avoid_note}\n\nSentence: {sent}",
        }]
        prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        encoded = tok(prompt, return_tensors="pt").to("cuda")
        prompt_len = encoded["input_ids"].shape[1]
        processor = PhonemeConstraintLogitsProcessor(tok, blocked_patterns, decoder_start_len=prompt_len)

        import time
        t0 = time.perf_counter()
        with torch.no_grad():
            out = model.generate(
                **encoded, max_new_tokens=80, num_beams=4, num_return_sequences=min(k, 4),
                no_repeat_ngram_size=3, early_stopping=True,
                logits_processor=LogitsProcessorList([processor]),
                pad_token_id=tok.eos_token_id,
            )
        latency = time.perf_counter() - t0
        candidates = []
        for seq in out:
            text = tok.decode(seq[prompt_len:], skip_special_tokens=True).strip()
            if text and text not in candidates:
                candidates.append(text)
        results.append({
            "run_id": run["id"], "original": sent,
            "blocked_words": blocked_words, "blocked_patterns": blocked_patterns,
            "candidates": candidates[:k], "stats": {"beam_kills": processor.kill_count},
            "latency_s": round(latency, 3),
        })
    del model
    torch.cuda.empty_cache()
    return results

In [ ]:
all_results = {}
for backbone in SEQ2SEQ_BACKBONES:
    print(f"=== {backbone} ===")
    all_results[backbone] = run_seq2seq_backbone(backbone, SUBSET)
    print(f"  done, {len(all_results[backbone])} runs")

print(f"=== {CAUSAL_BACKBONE} ===")
all_results[CAUSAL_BACKBONE] = run_causal_backbone(CAUSAL_BACKBONE, SUBSET)
print(f"  done, {len(all_results[CAUSAL_BACKBONE])} runs")

## 4. Hard mechanical checks — not judgment calls

Verify directly, don't trust the processor's own claim: does any surviving candidate actually still contain a blocked sound/word? This is checkable by code, same `phonetic.matches_any()` the live pipeline itself uses.

In [ ]:
import json
import re

def verify_leak_free(results):
    """Two different, both-useful counts, not conflated into one:
    - runs_with_any_leaking_candidate: at least one of the k candidates
      still contains a blocked word/pattern (informative but not fatal --
      the pipeline only needs ONE clean candidate per run).
    - runs_with_no_clean_candidate: none of the k candidates are clean --
      this is the one that actually matters for real-world usability.
    (An earlier version of this function incremented a single "n_leaked"
    counter once per LEAKING CANDIDATE while labeling it
    "runs_with_a_leak" -- a real bug, found by manually re-deriving the
    numbers against a real run's output. Fixed here to count per-run,
    both ways, explicitly.)"""
    n_runs = len(results)
    n_with_candidates = 0
    n_runs_with_any_leak = 0
    n_runs_with_no_clean_candidate = 0
    total_latency = 0.0
    for r in results:
        total_latency += r["latency_s"]
        if r["candidates"]:
            n_with_candidates += 1
        leaking_candidates = set()
        for cand in r["candidates"]:
            for w in re.findall(r"[A-Za-z][A-Za-z'-]*", cand):
                if any(bw.lower() == w.lower() for bw in r["blocked_words"]) or \
                   phonetic.matches_any(w, r["blocked_patterns"]):
                    leaking_candidates.add(cand)
                    break
        if leaking_candidates:
            n_runs_with_any_leak += 1
        n_clean = len(r["candidates"]) - len(leaking_candidates)
        if n_clean == 0:
            n_runs_with_no_clean_candidate += 1
    return {
        "n_runs": n_runs,
        "runs_with_any_candidate": n_with_candidates,
        "runs_with_any_leaking_candidate": n_runs_with_any_leak,
        "runs_with_no_clean_candidate": n_runs_with_no_clean_candidate,
        "mean_latency_s": round(total_latency / max(1, n_runs), 2),
    }

summary = {name: verify_leak_free(res) for name, res in all_results.items()}
print(json.dumps(summary, indent=2))
print("\nThe number that actually disqualifies a backbone is 'runs_with_no_clean_candidate' --\n"
      "a run with SOME leaking candidates but at least one clean one is still fine, since the\n"
      "real pipeline only needs to find one usable candidate per sentence.")

## 5. Save raw outputs for blind judging back in the main session

Meaning-preservation and naturalness are judgment calls, not mechanical checks — per this project's own discipline, those get judged blind (no metadata, same rubric as every prior phase) via a fresh Claude call in the main session, not scored here.

In [ ]:
import json
from pathlib import Path

output = {"mechanical_summary": summary, "raw_results": all_results}
Path("generation_backbone_comparison_results.json").write_text(json.dumps(output, indent=2))
print("saved generation_backbone_comparison_results.json")

try:
    from google.colab import files
    files.download("generation_backbone_comparison_results.json")
except ImportError:
    print("Not in Colab -- find the file in the working directory.")

## 6. What to do with this

1. Check the mechanical summary first (section 4) — a backbone with a high leak rate is disqualified regardless of how good its phrasing looks, no exception.
2. Bring `generation_backbone_comparison_results.json` back to the main session for blind quality judging (same rubric as every prior phase: meaning preservation, naturalness, grammaticality).
3. This is still Stage LR research — a better-performing backbone here is a candidate for further Stage LR work, not an authorized change to `main`'s frozen `rephrase.py`, per the freeze.